# Una CNN desde cero y un modelo preentrenado

**Sesión 3 · IA Intermedio · EstudIA 2026**

Vamos a clasificar fotos pequeñas en 10 clases (aviones, autos, pájaros, gatos…) de tres maneras:

1. Una **red convolucional (CNN)** que entrenamos desde cero en PyTorch.
2. La misma red, entrenada otra vez con **aumento de datos**.
3. Un modelo que otra persona ya entrenó, descargado de **Hugging Face**.

Al final comparamos los tres con las mismas imágenes de prueba.

**El recorrido.**

| Parte | Qué hacemos |
|---|---|
| 1 | Una convolución a mano |
| 2 | Los datos: CIFAR-10 |
| 3 | La CNN en PyTorch |
| 4 | Entrenar sin aumento de datos |
| 5 | Entrenar con aumento de datos |
| 6 | Lo que aprendió la red |
| 7 | Un modelo preentrenado de Hugging Face |
| 8 | Comparar los tres |

**Antes de empezar.**

1. *Archivo › Guardar una copia en Drive*.
2. Activa la GPU: *Entorno de ejecución › Cambiar tipo de entorno de ejecución › GPU T4 › Guardar*.
   Con GPU, cada entrenamiento tarda un par de minutos. Sin GPU tardaría mucho más: en ese caso el
   notebook carga los pesos que entrené yo con GPU y sigues igual el recorrido.
3. Ejecuta la celda de preparación (▶ o `Shift + Enter`).

In [ ]:
# Preparación: librerías, GPU y semillas. Ejecútala una vez.
import os, json, time, urllib.request
ARCHIVOS = os.environ.get("ESTUDIA_ARCHIVOS", "https://storage.googleapis.com/estud-ia-course-estudia-lab/s03")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

GPU = torch.cuda.is_available()
DISPOSITIVO = torch.device("cuda" if GPU else "cpu")

if GPU:
    print("Listo. GPU:", torch.cuda.get_device_name(0), "· PyTorch", torch.__version__)
else:
    print("Listo, pero sin GPU. Actívala en Entorno de ejecución › Cambiar tipo de entorno de ejecución › GPU T4\n"
          "y vuelve a ejecutar esta celda. Si Colab no te da una, el notebook carga los pesos que entrené con GPU.")

In [ ]:
# @title Funciones auxiliares: descargar archivos, escribir números y dibujar curvas
def descargar(nombre):
    """Baja un archivo de la carpeta pública del curso (solo la primera vez) y devuelve su ruta."""
    ruta = os.path.join("archivos_curso", nombre)
    if not os.path.exists(ruta):
        os.makedirs(os.path.dirname(ruta), exist_ok=True)
        urllib.request.urlretrieve(f"{ARCHIVOS}/{nombre}", ruta)
    return ruta

def miles(n):
    """12345 -> "12.345", como escribimos los números en español."""
    return f"{n:,}".replace(",", ".")

def curvas(historia, titulo):
    """Pérdida y exactitud por época, en entrenamiento y en validación."""
    epocas = range(1, len(historia["loss"]) + 1)
    fig, ejes = plt.subplots(1, 2, figsize=(11, 3.6))
    for eje, metrica, nombre in ((ejes[0], "loss", "Pérdida"), (ejes[1], "accuracy", "Exactitud")):
        eje.plot(epocas, historia[metrica], label="Entrenamiento", color="#000000", lw=2)
        eje.plot(epocas, historia["val_" + metrica], label="Validación", color="#CB318E", lw=2)
        eje.set_xlabel("Época"); eje.set_title(nombre); eje.grid(alpha=0.3)
        eje.spines[["top", "right"]].set_visible(False)
    mejor = int(np.argmin(historia["val_loss"]))
    ejes[0].axvline(mejor + 1, color="#7FC200", ls="--", lw=1.5, label=f"Menor pérdida de validación (época {mejor + 1})")
    ejes[0].legend(frameon=False, fontsize=9)
    fig.suptitle(titulo, fontweight="bold"); fig.tight_layout(); plt.show()

def cargar_entrenada(modelo, nombre):
    """Sin GPU: carga los pesos y la historia de un entrenamiento que hice con GPU."""
    modelo.load_state_dict(torch.load(descargar(f"{nombre}.pt"), map_location=DISPOSITIVO))
    with open(descargar(f"{nombre}.json"), encoding="utf-8") as f:
        historia = json.load(f)
    print(f"Sin GPU: cargué los pesos de “{nombre}”, entrenados con GPU durante {len(historia['loss'])} épocas.")
    return historia

---
## 1. Una convolución a mano

Para un computador, una foto en escala de grises es una tabla de números: uno por píxel, de 0 (negro)
a 1 (blanco). Una **convolución** recorre esa tabla con un **filtro**, una tablita de 3 × 3 números.
En cada posición multiplica los 9 píxeles por los 9 números del filtro y suma: ese es un píxel de la
imagen de salida.

Empecemos con un caso pequeño: una franja oscura (ceros) al lado de una franja clara (unos).

In [ ]:
parche = torch.tensor([[0, 0, 0, 1, 1, 1]] * 5, dtype=torch.float32)
bordes_verticales = torch.tensor([[-1, 0, 1],
                                  [-2, 0, 2],
                                  [-1, 0, 1]], dtype=torch.float32)

# F.conv2d espera 4 dimensiones: (imágenes, canales, alto, ancho) y (filtros, canales, alto, ancho).
salida = F.conv2d(parche[None, None], bordes_verticales[None, None])
print("Entrada:\n", parche.int().numpy())
print("\nSalida:\n", salida[0, 0].int().numpy())

La salida vale 0 donde todo es igual y crece justo donde la imagen pasa de oscuro a claro. Ese filtro,
el de Sobel, es un **detector de bordes verticales**. Ahora, el mismo cálculo sobre una foto real,
con cuatro filtros distintos.

In [ ]:
foto_gris = Image.open(descargar("img/metrocable.jpg")).convert("L")
x = torch.tensor(np.asarray(foto_gris), dtype=torch.float32)[None, None] / 255

FILTROS = {
    "Bordes verticales": bordes_verticales,
    "Bordes horizontales": bordes_verticales.T,
    "Desenfoque (9 × 9)": torch.full((9, 9), 1 / 81),
    "Más nitidez": torch.tensor([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=torch.float32),
}
fig, ejes = plt.subplots(1, 5, figsize=(19, 3.4))
ejes[0].imshow(foto_gris, cmap="gray"); ejes[0].set_title("Original")
for eje, (nombre, filtro) in zip(ejes[1:], FILTROS.items()):
    y = F.conv2d(x, filtro[None, None], padding=filtro.shape[0] // 2)[0, 0]
    if "Bordes" in nombre:
        eje.imshow(y.abs(), cmap="gray", vmax=float(y.abs().max()) / 3)   # más contraste para ver los bordes
    else:
        eje.imshow(y.clamp(0, 1), cmap="gray")
    eje.set_title(nombre)
for eje in ejes:
    eje.axis("off")
plt.show()
print("Foto: Bernard Gagnon, “Línea K (Medellín Metrocable) 03”, CC BY-SA 4.0, Wikimedia Commons.")

**Para pensar.**

- ¿Qué filtro resalta mejor los cables? ¿Y las casas de la ladera?
- Estos filtros los diseñó una persona. ¿Qué filtros servirían para distinguir un gato de un perro?

---
## 2. Los datos: CIFAR-10

CIFAR-10 tiene 60.000 fotos a color de 32 × 32 píxeles en 10 clases, 6.000 por clase. Es uno de los
conjuntos de datos más usados para probar redes de visión. Lo descargamos de Hugging Face con la
librería `datasets`. Tarda menos de un minuto.

In [ ]:
from datasets import load_dataset

cifar = load_dataset("uoft-cs/cifar10")
CLASES = cifar["train"].features["label"].names
NOMBRES = {"airplane": "avión", "automobile": "automóvil", "bird": "pájaro", "cat": "gato", "deer": "venado",
           "dog": "perro", "frog": "rana", "horse": "caballo", "ship": "barco", "truck": "camión"}

def a_tensores(parte):
    """Fotos -> tensor de enteros (n, 3 canales, 32, 32) y etiquetas -> tensor de enteros."""
    x = np.stack([np.asarray(foto) for foto in parte["img"]])
    return torch.from_numpy(x).permute(0, 3, 1, 2).contiguous(), torch.tensor(parte["label"])

X_todo, y_todo = a_tensores(cifar["train"])
X_prueba, y_prueba = a_tensores(cifar["test"])
print(f"{miles(len(X_todo))} fotos de entrenamiento y {miles(len(X_prueba))} de prueba. Forma de un lote: {tuple(X_todo.shape)}")

In [ ]:
fig, ejes = plt.subplots(6, 10, figsize=(13, 8.2))
for c in range(10):
    for f, i in enumerate(torch.where(y_prueba == c)[0][:6]):
        ejes[f, c].imshow(X_prueba[i].permute(1, 2, 0))
        ejes[f, c].axis("off")
    ejes[0, c].set_title(NOMBRES[CLASES[c]])
plt.tight_layout(); plt.show()

### 2.1 Las particiones

Para que cada entrenamiento tarde un par de minutos, entrenamos con **10.000** fotos. Separamos otras
**5.000** para validación y dejamos las **10.000** de prueba para el final. Las estadísticas para
normalizar los colores salen solo del entrenamiento, como en la sesión 2.

Después pasamos los datos a la GPU: cada lote ya está ahí cuando la red lo necesita.

In [ ]:
orden = torch.randperm(len(X_todo), generator=torch.Generator().manual_seed(42))
X_val, y_val = X_todo[orden[:5000]], y_todo[orden[:5000]]
X_ent, y_ent = X_todo[orden[5000:15000]], y_todo[orden[5000:15000]]

X_ent, y_ent, X_val, y_val = (t.to(DISPOSITIVO) for t in (X_ent, y_ent, X_val, y_val))
X_prueba_d, y_prueba_d = X_prueba.to(DISPOSITIVO), y_prueba.to(DISPOSITIVO)

MEDIA = (X_ent.float() / 255).mean(dim=(0, 2, 3), keepdim=True)
DESV = (X_ent.float() / 255).std(dim=(0, 2, 3), keepdim=True)

def normalizar(x):
    """Enteros de 0 a 255 -> números con media 0 y desviación 1 en cada canal de color."""
    return (x.float() / 255 - MEDIA) / DESV

print(f"Entrenamiento: {miles(len(X_ent))} · Validación: {miles(len(X_val))} · Prueba: {miles(len(X_prueba))} · en {DISPOSITIVO}")

---
## 3. La CNN en PyTorch

En Keras armábamos la red con `Sequential` y la entrenábamos con `fit`. En PyTorch también hay
`nn.Sequential`, pero el ciclo de entrenamiento lo escribimos nosotros: así vemos qué pasa en cada paso.

Nuestra red tiene tres **bloques**. Cada bloque aplica dos convoluciones de 3 × 3 y después un
**max pooling**, que reduce la imagen a la mitad quedándose con el valor más alto de cada cuadro de
2 × 2. Al final, una **cabeza** densa, como el MLP de la sesión 2, con una salida por clase.

In [ ]:
def bloque(entrada, salida):
    """Dos convoluciones de 3 × 3 con normalización por lotes y ReLU, y un max pooling."""
    return nn.Sequential(
        nn.Conv2d(entrada, salida, kernel_size=3, padding=1), nn.BatchNorm2d(salida), nn.ReLU(),
        nn.Conv2d(salida, salida, kernel_size=3, padding=1), nn.BatchNorm2d(salida), nn.ReLU(),
        nn.MaxPool2d(2),
    )

def crear_cnn():
    torch.manual_seed(0)
    return nn.Sequential(
        bloque(3, 32),        # 32 × 32 -> 16 × 16, con 32 filtros
        bloque(32, 64),       # 16 × 16 -> 8 × 8, con 64 filtros
        bloque(64, 128),      # 8 × 8 -> 4 × 4, con 128 filtros
        nn.Flatten(),         # 128 × 4 × 4 = 2.048 números
        nn.Linear(2048, 256), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(256, len(CLASES)),   # una salida por clase (la softmax va dentro de la pérdida)
    ).to(DISPOSITIVO)

cnn = crear_cnn()
x = normalizar(X_ent[:8])
print("Entrada:", tuple(x.shape))
for i, capa in enumerate(cnn):
    x = capa(x)
    if i < 3 or i == len(cnn) - 1:
        print(f"Después de {'el bloque ' + str(i + 1) if i < 3 else 'la cabeza'}: {tuple(x.shape)}")
print(f"\nParámetros: {miles(sum(p.numel() for p in cnn.parameters()))}")

Fíjate en la primera convolución: 32 filtros de 3 × 3 sobre 3 canales de color son
3 × 3 × 3 × 32 + 32 = **896** pesos, y cada filtro recorre toda la imagen. Una capa densa de 256
neuronas sobre los 3.072 números de la foto necesitaría **786.688**.

### 3.1 El ciclo de entrenamiento

Por cada lote de 128 fotos hacemos cuatro pasos: **predecir**, **medir el error** con la entropía
cruzada, **calcular los gradientes** (`backward`) y **ajustar los pesos** (`step`). Eso es lo que hace
`fit` en Keras. Al final de cada época medimos la pérdida y la exactitud en validación.

In [ ]:
@torch.no_grad()
def evaluar(modelo, X, y, lote=1000):
    """Pérdida y exactitud promedio, sin ajustar pesos."""
    modelo.eval()
    perdida, aciertos = 0.0, 0
    for i in range(0, len(X), lote):
        salida = modelo(normalizar(X[i:i + lote]))
        perdida += F.cross_entropy(salida, y[i:i + lote], reduction="sum").item()
        aciertos += (salida.argmax(1) == y[i:i + lote]).sum().item()
    return perdida / len(X), aciertos / len(X)

def entrenar(modelo, epocas, aumentar=None, lote=128):
    """El ciclo de entrenamiento. Devuelve la historia de pérdida y exactitud por época."""
    torch.manual_seed(0)
    optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)
    programa = torch.optim.lr_scheduler.CosineAnnealingLR(optimizador, T_max=epocas)  # baja la tasa poco a poco
    historia = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}
    inicio = time.time()
    for epoca in range(1, epocas + 1):
        modelo.train()                                   # dropout y normalización por lotes en modo entrenamiento
        orden = torch.randperm(len(X_ent), device=DISPOSITIVO)
        suma, aciertos = 0.0, 0
        for i in range(0, len(orden), lote):
            idx = orden[i:i + lote]
            x, y = X_ent[idx], y_ent[idx]
            if aumentar is not None:
                x = aumentar(x)                          # una versión alterada de cada foto
            salida = modelo(normalizar(x))               # 1. predecir
            perdida = F.cross_entropy(salida, y)         # 2. medir el error
            optimizador.zero_grad()
            perdida.backward()                           # 3. calcular los gradientes
            optimizador.step()                           # 4. ajustar los pesos
            suma += perdida.item() * len(y)
            aciertos += (salida.argmax(1) == y).sum().item()
        programa.step()
        val_perdida, val_exactitud = evaluar(modelo, X_val, y_val)
        for k, v in zip(historia, (suma / len(X_ent), aciertos / len(X_ent), val_perdida, val_exactitud)):
            historia[k].append(v)
        if epoca == 1 or epoca % 5 == 0:
            print(f"Época {epoca:2d}/{epocas} · entrenamiento: pérdida {historia['loss'][-1]:.3f}, "
                  f"exactitud {historia['accuracy'][-1]:.1%} · validación: pérdida {val_perdida:.3f}, "
                  f"exactitud {val_exactitud:.1%} · {time.time() - inicio:.0f} s")
    return historia

---
## 4. Entrenar sin aumento de datos

40 épocas: la red ve cada una de las 10.000 fotos 40 veces.

In [ ]:
EPOCAS = 40
cnn_sin = crear_cnn()
if GPU:
    historia_sin = entrenar(cnn_sin, EPOCAS)
else:
    historia_sin = cargar_entrenada(cnn_sin, "cnn_sin_aumento")
curvas(historia_sin, "CNN sin aumento de datos")

**Para pensar.**

- ¿En qué época es más baja la pérdida de validación? ¿Qué pasa después?
- La exactitud de entrenamiento se acerca a 100 %. ¿Qué aprendió la red?

---
## 5. Aumento de datos

En cada lote le mostramos a la red una versión distinta de cada foto: la mitad, al azar, volteadas
como en un espejo, y todas corridas hasta 4 píxeles en cualquier dirección. Un gato volteado sigue
siendo un gato, así que la etiqueta no cambia. La red casi nunca ve dos veces la misma imagen exacta
y le cuesta memorizar píxeles: tiene que aprender lo que hace que un gato sea un gato.

El aumento se usa **solo en el entrenamiento**. Validación y prueba quedan como están.

In [ ]:
def aumentar(x, desplazamiento=4):
    """Una versión distinta de cada foto del lote: espejo al azar y un corrimiento de hasta 4 píxeles."""
    n, _, alto, ancho = x.shape
    espejo = torch.rand(n, device=x.device) < 0.5
    x = torch.where(espejo[:, None, None, None], x.flip(3), x)
    # Corrimiento: ampliamos el borde reflejando la foto y recortamos una ventana movida al azar.
    x = F.pad(x, [desplazamiento] * 4, mode="reflect")
    dy, dx = torch.randint(0, 2 * desplazamiento + 1, (2, n), device=x.device)
    filas = dy[:, None] + torch.arange(alto, device=x.device)
    columnas = dx[:, None] + torch.arange(ancho, device=x.device)
    lotes = torch.arange(n, device=x.device)[:, None, None]
    return x.permute(0, 2, 3, 1)[lotes, filas[:, :, None], columnas[:, None, :]].permute(0, 3, 1, 2)

i = int(torch.where(y_ent == CLASES.index("cat"))[0][0])
copias = aumentar(X_ent[i:i + 1].repeat(8, 1, 1, 1))
fig, ejes = plt.subplots(1, 9, figsize=(13, 1.8))
ejes[0].imshow(X_ent[i].permute(1, 2, 0).cpu()); ejes[0].set_title("Original")
for eje, c in zip(ejes[1:], copias):
    eje.imshow(c.permute(1, 2, 0).cpu())
for eje in ejes:
    eje.axis("off")
plt.show()

In [ ]:
cnn_con = crear_cnn()
if GPU:
    historia_con = entrenar(cnn_con, EPOCAS, aumentar=aumentar)
else:
    historia_con = cargar_entrenada(cnn_con, "cnn_con_aumento")
curvas(historia_con, "CNN con aumento de datos")

In [ ]:
resumen = pd.DataFrame({
    "exactitud de entrenamiento (última época)": [historia_sin["accuracy"][-1], historia_con["accuracy"][-1]],
    "exactitud de validación (última época)": [historia_sin["val_accuracy"][-1], historia_con["val_accuracy"][-1]],
    "pérdida de validación mínima": [min(historia_sin["val_loss"]), min(historia_con["val_loss"])],
    "pérdida de validación final": [historia_sin["val_loss"][-1], historia_con["val_loss"][-1]],
}, index=["Sin aumento", "Con aumento"])
resumen.round(3)

**Para pensar.**

- Con aumento, la exactitud de entrenamiento es más baja. ¿Por qué eso no es una mala noticia?
- ¿Qué otras alteraciones conservarían la clase de estas fotos? ¿Cuáles no? Piensa en voltear de arriba abajo.

---
## 6. Lo que aprendió la red

En la parte 1 elegimos los números de cada filtro. La CNN los aprende sola. Estos son los 32 filtros
de su primera capa, dibujados como pequeñas imágenes a color, y lo que producen sobre una foto.

In [ ]:
pesos = cnn_con[0][0].weight.detach().cpu()                       # (32 filtros, 3 colores, 3, 3)
pesos = (pesos - pesos.amin(dim=(1, 2, 3), keepdim=True)) / (pesos.amax(dim=(1, 2, 3), keepdim=True) - pesos.amin(dim=(1, 2, 3), keepdim=True))
fig, ejes = plt.subplots(2, 16, figsize=(13, 1.9))
for eje, w in zip(ejes.flat, pesos):
    eje.imshow(w.permute(1, 2, 0)); eje.axis("off")
plt.suptitle("Los 32 filtros de la primera capa"); plt.show()

cnn_con.eval()
with torch.no_grad():
    mapas = cnn_con[0][:3](normalizar(X_prueba_d[:1]))[0].cpu()  # primera convolución + normalización + ReLU
fig, ejes = plt.subplots(1, 9, figsize=(13, 1.8))
ejes[0].imshow(X_prueba[0].permute(1, 2, 0)); ejes[0].set_title("Foto")
for k, eje in enumerate(ejes[1:]):
    eje.imshow(mapas[k], cmap="gray"); eje.set_title(f"Filtro {k + 1}")
for eje in ejes:
    eje.axis("off")
plt.show()

---
## 7. Un modelo preentrenado de Hugging Face

[`aaraki/vit-base-patch16-224-in21k-finetuned-cifar10`](https://huggingface.co/aaraki/vit-base-patch16-224-in21k-finetuned-cifar10)
es un **Vision Transformer (ViT)** de Google. Primero lo entrenaron con unos 14 millones de imágenes de
ImageNet-21k; después, otra persona lo ajustó con las 50.000 fotos de entrenamiento de CIFAR-10.
Tiene unos 86 millones de parámetros, cien veces más que nuestra CNN.

### 7.1 La forma corta: `pipeline`

`pipeline` descarga el modelo, prepara cada imagen como la espera el modelo (la agranda a
224 × 224 y normaliza sus colores) y devuelve las clases más probables. La primera vez tarda un poco:
baja unos 350 MB.

In [ ]:
from transformers import pipeline

MODELO_HF = "aaraki/vit-base-patch16-224-in21k-finetuned-cifar10"
clasificador = pipeline("image-classification", model=MODELO_HF, device=0 if GPU else -1)

fotos = [(cifar["test"][int(i)]["img"], "CIFAR-10: " + NOMBRES[CLASES[int(y_prueba[i])]]) for i in (0, 1, 3, 12)]
fotos += [(Image.open(descargar(f"img/{f}")).convert("RGB"), titulo)
          for f, titulo in (("gato.jpg", "Foto: gato"), ("tucan.jpg", "Foto: tucán"), ("arepas.jpg", "Foto: arepas"))]
resultados = clasificador([f for f, _ in fotos], top_k=3)

fig, ejes = plt.subplots(1, len(fotos), figsize=(17, 3.6))
for eje, (foto, titulo), r in zip(ejes, fotos, resultados):
    eje.imshow(foto); eje.axis("off")
    eje.set_title(f"{titulo}\n" + "\n".join(f"{NOMBRES[p['label']]} {p['score']:.0%}" for p in r), fontsize=9)
plt.show()
print("Fotos: Alvesgaspar (gato, CC BY-SA 3.0), Bernard Dupont (tucán, CC BY-SA 2.0) e Instituto de Turismo del Meta "
      "(arepas, CC BY-SA 4.0), Wikimedia Commons.")

**Para pensar.**

- ¿Por qué el modelo le pone una de las 10 clases a las arepas, si no son ninguna? Acuérdate de la softmax de la sesión 2.

### 7.2 Evaluarlo con todas las fotos de prueba

Para evaluar miles de fotos usamos las dos piezas que `pipeline` esconde: el **procesador**, que
prepara las imágenes, y el **modelo**. Con GPU evaluamos las 10.000 fotos de prueba en alrededor de un
minuto. Sin GPU usamos las primeras 300, y comparamos los tres modelos con esas mismas 300.

In [ ]:
from transformers import AutoImageProcessor, AutoModelForImageClassification

procesador = AutoImageProcessor.from_pretrained(MODELO_HF)
vit = AutoModelForImageClassification.from_pretrained(MODELO_HF).to(DISPOSITIVO).eval()
a_cifar = {i: CLASES.index(nombre) for i, nombre in vit.config.id2label.items()}  # misma clase, mismo número

N = len(X_prueba) if GPU else 300
pred_vit, inicio = [], time.time()
with torch.no_grad():
    for i in range(0, N, 250):
        entrada = procesador(images=cifar["test"][i:min(i + 250, N)]["img"], return_tensors="pt").to(DISPOSITIVO)
        pred_vit += [a_cifar[int(k)] for k in vit(**entrada).logits.argmax(-1)]
pred_vit = torch.tensor(pred_vit)
print(f"ViT: {miles(N)} fotos en {time.time() - inicio:.0f} s · exactitud {(pred_vit == y_prueba[:N]).float().mean():.1%}")

---
## 8. Comparar los tres

Los tres modelos, con las mismas fotos de prueba que ninguno usó para aprender.

In [ ]:
@torch.no_grad()
def predecir(modelo, X, lote=1000):
    modelo.eval()
    return torch.cat([modelo(normalizar(X[i:i + lote])).argmax(1).cpu() for i in range(0, len(X), lote)])

pred = {"CNN sin aumento": predecir(cnn_sin, X_prueba_d[:N]),
        "CNN con aumento": predecir(cnn_con, X_prueba_d[:N]),
        "ViT preentrenado": pred_vit}
y_n = y_prueba[:N]
comparacion = pd.DataFrame({
    "exactitud en prueba": [(p == y_n).float().mean().item() for p in pred.values()],
    "parámetros": [miles(sum(p.numel() for p in cnn_sin.parameters())), miles(sum(p.numel() for p in cnn_con.parameters())),
                   miles(sum(p.numel() for p in vit.parameters()))],
    "imágenes con las que aprendió": ["10.000", "10.000 (con aumento)", "unos 14 millones + 50.000"],
}, index=list(pred))
display(comparacion.style.format({"exactitud en prueba": "{:.1%}"}))

por_clase = pd.DataFrame({m: [(p[y_n == c] == c).float().mean().item() for c in range(10)] for m, p in pred.items()},
                         index=[NOMBRES[c] for c in CLASES])
ax = por_clase.plot.bar(figsize=(12, 3.6), color=["#5B6170", "#CB318E", "#7FC200"], width=0.8)
ax.set_ylabel("Exactitud"); ax.set_ylim(0, 1); ax.set_title("Exactitud por clase")
ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.12))
ax.spines[["top", "right"]].set_visible(False); plt.xticks(rotation=0); plt.show()

**Para pensar.**

- ¿Qué clases le cuestan más a nuestra CNN? ¿Por qué crees que gato y perro se confunden?
- El ViT gana por mucho. ¿Qué tuvo él que nuestra CNN no tuvo?
- Si tu proyecto tiene 500 fotos propias, ¿empezarías con una red desde cero o con un modelo preentrenado?

---
## Cierre

- Una convolución recorre la imagen con un filtro; un filtro bien elegido encuentra bordes.
- Una CNN aprende sus propios filtros, capa por capa.
- Sin aumento de datos, la red memorizó las fotos de entrenamiento. Con aumento, generalizó mejor.
- Un modelo preentrenado con millones de imágenes superó a las dos CNN, y nosotros solo lo descargamos.

**Fuentes.** Datos: Krizhevsky, A. (2009). *Learning Multiple Layers of Features from Tiny Images*
(CIFAR-10), copia en Hugging Face `uoft-cs/cifar10`. Modelo: `aaraki/vit-base-patch16-224-in21k-finetuned-cifar10`
(licencia Apache 2.0), a partir de `google/vit-base-patch16-224-in21k` (Dosovitskiy y otros, 2021).
Fotos de Wikimedia Commons: Bernard Gagnon (Metrocable, CC BY-SA 4.0), Alvesgaspar (gato, CC BY-SA 3.0),
Bernard Dupont (tucán, CC BY-SA 2.0), Instituto de Turismo del Meta, foto de Constantino Castelblanco
(arepas, CC BY-SA 4.0).